# Lab: Enterprise RAG Pattern

## Overview

Retrieval-Augmented Generation (RAG) grounds an LLM in real documents rather than training-time knowledge. This lab builds a production-grade RAG pipeline: document ingestion, chunking, embedding, vector indexing, hybrid retrieval, re-ranking, and integration with a LangChain agent

## What You Will Build

A governed RAG agent that retrieves from a policy and clinical document corpus, applies metadata filtering for access control, uses hybrid dense+keyword search, and exposes retrieval as a `@tool` so the agent decides when to retrieve.

## Prerequisites

- Azure OpenAI resource with a GPT-4o deployment and a `text-embedding-ada-002`deployment.
- Python 3.10+. All packages installed in Cell 2.

In [1]:
# ── Install all dependencies ──────────────────────────────────────────────────
%pip install langchain langchain-openai langchain-community langgraph azure-identity \
             openai python-dotenv tiktoken faiss-cpu langchain-text-splitters azure-search-documents --upgrade --quiet

In [ ]:
from pathlib import Path

env_content = """
AZURE_OPENAI_ENDPOINT=https://<TBD>.cognitiveservices.azure.com
AZURE_OPENAI_API_KEY=
AZURE_OPENAI_DEPLOYMENT_NAME=gpt-4o
AZURE_OPENAI_API_VERSION=2024-12-01-preview
AZURE_OPENAI_EMBEDDING_DEPLOYMENT=text-embedding-ada-002
AZURE_SEARCH_ENDPOINT=https://<TBD>.search.windows.net
AZURE_SEARCH_KEY=<TBD>
AZURE_SEARCH_INDEX=enterprise-rag-data
"""

# Create .env file in current directory
env_path = Path(".env")
env_path.write_text(env_content.strip())

print(".env file created successfully.")
print(env_path.resolve())

.env file created successfully.
/content/.env


In [3]:
import os, uuid
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(override=True)

# ── LLM ───────────────────────────────────────────────────────────────────────
llm = AzureChatOpenAI(
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature=0, max_tokens=4096,
)

# ── Embeddings ────────────────────────────────────────────────────────────────
# Update azure_deployment to your embedding model deployment name.
# Common values: 'text-embedding-3-large' or 'text-embedding-ada-002'
embeddings = AzureOpenAIEmbeddings(
    azure_deployment=os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
)

print("LLM and embeddings initialised.")

LLM and embeddings initialised.


## Section 1 — RAG Architecture

### Why RAG?

An LLM's training knowledge has a cutoff date, cannot access proprietary documents, and hallucinates when asked about specifics it was not trained on. RAG solves all three by splitting the problem: the LLM handles reasoning and language generation; a vector index handles factual retrieval from authoritative sources.

### The Four-Stage Pipeline

**Stage 1 — Ingestion:** Source documents (PDFs, regulatory filings, clinical guidelines, internal wikis) are loaded and split into chunks. Chunk size and overlap are the primary quality levers at this stage.

**Stage 2 — Embedding:** Each chunk is converted to a dense vector using an embedding model. Azure OpenAI `text-embedding-3-large` produces 3072-dimensional vectors and outperforms `ada-002` on domain-specific corpora. Embedding is a one-time cost per document; only new or updated documents need re-embedding.

**Stage 3 — Indexing:** Vectors are stored in a searchable index. In this lab: FAISS (in-process, no infrastructure). In production: Azure AI Search with HNSW approximate nearest-neighbour indexing, which scales to billions of vectors with sub-100ms query latency.

**Stage 4 — Retrieval:** At query time, the query is embedded and the index returns the top-k most similar chunks. The chunks are injected into the agent's context as grounding evidence.

### Chunking Strategy

Chunking is the most impactful decision in a RAG system:

- **Fixed-size with overlap** — 512 tokens, 10% overlap. Simple and effective for most corpora. Overlap prevents context loss at chunk boundaries.
- **Recursive character splitting** — splits on paragraph → sentence → word boundaries in order. Preserves semantic coherence better than fixed-size splitting.
- **Semantic chunking** — splits when the embedding similarity between consecutive sentences drops below a threshold. Highest quality; highest cost.

For financial and clinical documents, recursive splitting at 512 tokens with 10% overlap is the recommended starting point.

### Hybrid Search

Pure vector search can miss documents that use precise terminology (drug names, ticker symbols, regulation codes) without those exact terms appearing in the training distribution of the embedding model. Hybrid search combines:

- **Dense retrieval** — vector similarity (semantic meaning).
- **BM25 keyword retrieval** — exact term frequency matching.

Results from both are merged using Reciprocal Rank Fusion (RRF). Azure AI Search supports hybrid search natively. In FAISS (this lab), BM25 is simulated with a keyword filter applied as a post-processing step.

In [4]:
import os
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# ── Create data folder and write source documents as .txt files ───────────────
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

RAW_DOCUMENTS = [
    {"filename": "basel3_capital_rules.txt",
     "text": "Basel III requires banks to maintain a minimum Common Equity Tier 1 (CET1) capital ratio of 4.5% of risk-weighted assets. The Tier 1 capital ratio must be at least 6%. Banks must also hold a capital conservation buffer of 2.5%, bringing the effective CET1 floor to 7.0%. Systemically important institutions face additional surcharges of 1-3.5%.",
     "source": "basel3_capital_rules", "doc_type": "regulation", "access_level": "public"},

    {"filename": "dodd_frank_165.txt",
     "text": "Dodd-Frank Section 165 mandates that bank holding companies with assets exceeding $250 billion maintain a liquidity coverage ratio (LCR) of at least 100%. The LCR requires institutions to hold sufficient high-quality liquid assets (HQLA) to cover 30 days of net cash outflows under a stress scenario.",
     "source": "dodd_frank_165", "doc_type": "regulation", "access_level": "public"},

    {"filename": "hipaa_security_rule.txt",
     "text": "HIPAA Security Rule requires covered entities to implement administrative, physical, and technical safeguards for electronic protected health information (ePHI). Access controls must restrict ePHI access to authorised users only. Audit controls must record and examine activity in systems containing ePHI. Integrity controls must prevent unauthorised alteration or destruction of ePHI.",
     "source": "hipaa_security_rule", "doc_type": "regulation", "access_level": "public"},

    {"filename": "ada_2024_standards.txt",
     "text": "ADA 2024 Standards of Care: The recommended HbA1c target for most non-pregnant adults with type 2 diabetes is below 7.0%. First-line pharmacotherapy is metformin combined with lifestyle modification. If cardiovascular disease or high cardiovascular risk is present, add a GLP-1 receptor agonist or SGLT-2 inhibitor regardless of HbA1c level. SGLT-2 inhibitors are preferred when heart failure or chronic kidney disease is present.",
     "source": "ada_2024_standards", "doc_type": "clinical_guideline", "access_level": "public"},

    {"filename": "esc_2020_af.txt",
     "text": "ESC 2020 Atrial Fibrillation Guidelines: Stroke risk must be assessed using the CHA2DS2-VASc score. Anticoagulation is recommended for men with a score of 2 or higher and women with a score of 3 or higher. Non-vitamin K oral anticoagulants (NOACs) are preferred over warfarin for stroke prevention in AF unless the patient has moderate-to-severe mitral stenosis or a mechanical heart valve.",
     "source": "esc_2020_af", "doc_type": "clinical_guideline", "access_level": "public"},

    {"filename": "internal_policy_IB-2024-03.txt",
     "text": "Internal Investment Policy IB-2024-03: Analysts are prohibited from initiating positions in any company currently under regulatory investigation without written approval from the Chief Compliance Officer. All trades in pharmaceutical stocks exceeding $5M must be reviewed by both the sector analyst and the clinical advisory board before execution.",
     "source": "internal_policy_IB-2024-03", "doc_type": "internal_policy", "access_level": "restricted"},
]

# Write each document to data/ as a .txt file
# Metadata is stored as a sidecar .json file alongside each .txt
import json
for doc in RAW_DOCUMENTS:
    txt_path  = DATA_DIR / doc["filename"]
    meta_path = DATA_DIR / doc["filename"].replace(".txt", ".json")
    txt_path.write_text(doc["text"], encoding="utf-8")
    meta_path.write_text(json.dumps({
        "source":       doc["source"],
        "doc_type":     doc["doc_type"],
        "access_level": doc["access_level"],
    }, indent=2), encoding="utf-8")

print(f"Data folder: {DATA_DIR.resolve()}")
print(f"Files written: {len(list(DATA_DIR.glob('*.txt')))} .txt  +  {len(list(DATA_DIR.glob('*.json')))} .json")

# ── Load documents back from data/ folder ─────────────────────────────────────
from langchain_community.document_loaders import TextLoader

loaded_docs = []
for txt_path in sorted(DATA_DIR.glob("*.txt")):
    meta_path = txt_path.with_suffix(".json")
    metadata  = json.loads(meta_path.read_text(encoding="utf-8"))
    loader    = TextLoader(str(txt_path), encoding="utf-8")
    docs      = loader.load()
    for d in docs:
        d.metadata.update(metadata)   # attach source/doc_type/access_level
    loaded_docs.extend(docs)

print(f"\nLoaded {len(loaded_docs)} document(s) from {DATA_DIR}/")
for d in loaded_docs:
    print(f"  [{d.metadata['source']:<35}] access={d.metadata['access_level']}")

# ── Chunking ──────────────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
    length_function=len,
)

all_chunks = splitter.split_documents(loaded_docs)

print(f"\nDocuments: {len(loaded_docs)}  →  Chunks after splitting: {len(all_chunks)}")
for c in all_chunks:
    print(f"  [{c.metadata['source']:<35}] {len(c.page_content):>4} chars")

Data folder: /content/data
Files written: 6 .txt  +  6 .json

Loaded 6 document(s) from data/
  [ada_2024_standards                 ] access=public
  [basel3_capital_rules               ] access=public
  [dodd_frank_165                     ] access=public
  [esc_2020_af                        ] access=public
  [hipaa_security_rule                ] access=public
  [internal_policy_IB-2024-03         ] access=restricted

Documents: 6  →  Chunks after splitting: 7
  [ada_2024_standards                 ]  395 chars
  [ada_2024_standards                 ]   70 chars
  [basel3_capital_rules               ]  344 chars
  [dodd_frank_165                     ]  300 chars
  [esc_2020_af                        ]  390 chars
  [hipaa_security_rule                ]  385 chars
  [internal_policy_IB-2024-03         ]  348 chars


## Section 2 — Retrieval Quality and Access Control

### Metadata Filtering for Access Control

A naive RAG system retrieves the most similar chunks regardless of who is asking. In a multi-tenant enterprise, this is a security failure: a junior analyst should not receive chunks tagged `access_level: restricted` that are only approved for senior compliance staff.

The correct pattern is **pre-filter then retrieve**: apply a metadata filter before the vector search so the similarity search only runs over the subset of documents the requesting user is authorised to see. In FAISS this is done via `filter` in `similarity_search`. In Azure AI Search it is an OData `$filter` parameter applied before the HNSW traversal.

Access level is stored as a chunk metadata field at ingestion time. The retrieval function receives the caller's `access_level` and applies it as a hard constraint — not a soft ranking signal.

### Re-Ranking

The top-k results from a vector search are ordered by embedding similarity, which correlates with semantic relatedness but not necessarily with answer relevance. A **cross-encoder re-ranker** solves this: it takes each (query, chunk) pair and scores them jointly, which is slower but far more accurate than the bi-encoder used for retrieval.

Re-ranking pipeline:
1. Retrieve top-20 candidates by vector similarity (cheap, fast).
2. Re-rank the 20 candidates with a cross-encoder (slower, runs 20 inference calls).
3. Return the top-3 re-ranked results to the agent.

In production: use Azure AI Content Understanding or a Cohere Rerank endpoint. In this lab: a simple keyword-overlap score simulates re-ranking without an additional API call.

### Evaluation: Retrieval Quality Metrics

Before deploying a RAG system, measure retrieval quality on a golden dataset:

- **Hit rate @k** — fraction of queries where the correct document appears in the top-k results.
- **Mean Reciprocal Rank (MRR)** — average of 1/rank of the first correct result. Penalises systems that bury the correct answer.
- **Answer faithfulness** — does the generated answer contain only claims that can be traced to retrieved chunks? Measured by an LLM judge or NLI model.
- **Answer relevance** — does the generated answer actually address the question? Distinct from faithfulness.

In [5]:
from langchain_community.vectorstores import AzureSearch

vector_store = AzureSearch(
    azure_search_endpoint=os.getenv("AZURE_SEARCH_ENDPOINT"),
    azure_search_key=os.getenv("AZURE_SEARCH_KEY"),
    index_name=os.getenv("AZURE_SEARCH_INDEX"),
    embedding_function=embeddings.embed_query,
    additional_search_client_options={"api_version": "2024-07-01"},
)

vector_store.add_documents(all_chunks)
print(f"Done. {len(all_chunks)} chunks indexed.")

Done. 7 chunks indexed.


In [6]:
def _rerank(query: str, docs: list[Document], top_n: int = 3) -> list[Document]:
    """
    Lightweight keyword-overlap re-ranker.
    Production replacement: Cohere Rerank or Azure AI Content Understanding.
    """
    query_terms = set(query.lower().split())
    scored = []
    for doc in docs:
        doc_terms  = set(doc.page_content.lower().split())
        overlap    = len(query_terms & doc_terms) / max(len(query_terms), 1)
        scored.append((overlap, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored[:top_n]]


def retrieve(
    query:        str,
    access_level: str = "public",
    doc_type:     str = None,
    top_k:        int = 3,
) -> list:

    allowed_levels = {"public"} if access_level == "public" else {"public", "restricted"}

    # Retrieve more candidates than needed — filter manually after
    candidates = vector_store.similarity_search(query, k=20)

    # Apply access control and doc_type filter in Python
    filtered = [
        doc for doc in candidates
        if doc.metadata.get("access_level") in allowed_levels
        and (doc_type is None or doc.metadata.get("doc_type") == doc_type)
    ]

    return _rerank(query, filtered, top_n=top_k)


# ── Test retrieval ────────────────────────────────────────────────────────────
print("\nRetrieval test — Basel III capital query (public access):")
results = retrieve("minimum CET1 capital ratio requirements", access_level="public")
for r in results:
    print(f"  [{r.metadata['source']}] {r.page_content[:120]}...")

print("\nRetrieval test — internal policy (public access — should be excluded):")
results_pub = retrieve("pharmaceutical trade approval compliance", access_level="public")
sources = [r.metadata['source'] for r in results_pub]
print(f"  Sources returned: {sources}")
print(f"  Internal policy included: {'internal_policy_IB-2024-03' in sources}  ← should be False")

print("\nRetrieval test — internal policy (restricted access — should be included):")
results_res = retrieve("pharmaceutical trade approval compliance", access_level="restricted")
sources_res = [r.metadata['source'] for r in results_res]
print(f"  Sources returned: {sources_res}")
print(f"  Internal policy included: {'internal_policy_IB-2024-03' in sources_res}  ← should be True")


Retrieval test — Basel III capital query (public access):
  [basel3_capital_rules] Basel III requires banks to maintain a minimum Common Equity Tier 1 (CET1) capital ratio of 4.5% of risk-weighted assets...
  [basel3_capital_rules] Basel III requires banks to maintain a minimum Common Equity Tier 1 (CET1) capital ratio of 4.5% of risk-weighted assets...
  [dodd_frank_165] Dodd-Frank Section 165 mandates that bank holding companies with assets exceeding $250 billion maintain a liquidity cove...

Retrieval test — internal policy (public access — should be excluded):
  Sources returned: ['hipaa_security_rule', 'hipaa_security_rule', 'ada_2024_standards']
  Internal policy included: False  ← should be False

Retrieval test — internal policy (restricted access — should be included):
  Sources returned: ['internal_policy_IB-2024-03', 'internal_policy_IB-2024-03', 'hipaa_security_rule']
  Internal policy included: True  ← should be True


## Section 3 — Agent-Driven RAG vs Pre-Retrieval Injection

There are two architectures for integrating retrieval with an agent:

**Pre-retrieval injection (naive RAG):** Retrieve before the agent runs. Inject the top-k chunks into the system prompt. The agent always has the retrieved context, whether it needs it or not. Simple to implement; wasteful on tokens for queries that don't need retrieval; the agent cannot decide to retrieve more if the first retrieval was insufficient.

**Agent-driven RAG (tool-based RAG):** Expose retrieval as a `@tool`. The agent calls the tool when it determines retrieval is needed, with the exact query it formulates. This is more expensive per retrieval call (the tool call itself costs tokens) but the agent can: decide not to retrieve for questions it can answer from tool data alone, issue multiple targeted retrieval queries for multi-hop questions, and combine retrieval results with live tool data (stock prices + policy documents in a single turn).

**Recommendation:** use tool-based RAG for agents that also have live data tools (the case in this lab). Use pre-retrieval injection for dedicated Q&A systems where every query requires document grounding.

### Context Window Management

Each retrieved chunk consumes context window tokens. At 512 tokens per chunk and top-k=3, retrieval adds ~1500 tokens to every tool-call turn. For a multi-turn session with 10 turns and 2 retrieval calls per turn, retrieved context alone accounts for ~30,000 tokens. Mitigation strategies:

- **Summarise chunks before injection:** use a fast, cheap model to compress each chunk to 2-3 sentences before the agent sees it.
- **Strict top-k:** keep k=2 or k=3; resist the temptation to increase it for marginal quality gains.
- **Session-scoped deduplication:** if the same chunk was retrieved in a previous turn of the same session, do not re-inject it.

In [7]:
# ── Finance tools (needed by the RAG agent) ───────────────────────────────────
@tool
def get_stock_price(ticker: str) -> str:
    """Retrieve the current stock price for a given ticker symbol (e.g. AAPL, MSFT, JPM, PFE, JNJ)."""
    prices = {"AAPL": "189.45", "MSFT": "415.20", "JPM": "198.73",
              "GS":   "452.10", "JNJ":  "162.88", "PFE": "29.54"}
    t = ticker.upper().strip()
    return f"Current price of {t}: ${prices[t]} USD" if t in prices else f"Ticker {t} not found."

@tool
def get_interest_rate(rate_type: str) -> str:
    """Look up a benchmark interest rate. Supported: fed_funds, sofr, us_10yr_treasury, us_2yr_treasury."""
    rates = {"fed_funds": "5.25%", "sofr": "5.30%",
             "us_10yr_treasury": "4.48%", "us_2yr_treasury": "4.85%"}
    r = rates.get(rate_type.lower().strip())
    return f"{rate_type}: {r}" if r else f"Rate '{rate_type}' not recognised."


# ── RAG tool — agent calls this when it needs document grounding ──────────────
# Access level is fixed to 'public' here.
# In production: derive from the authenticated user's role/claims.
CALLER_ACCESS_LEVEL = "public"   # change to 'restricted' to unlock internal policy docs

@tool
def retrieve_policy(query: str) -> str:
    """
    Search the internal policy, regulation, and clinical guideline document index.
    Use this tool when the user asks about regulatory requirements, compliance thresholds,
    capital ratios, HIPAA rules, clinical treatment standards, or internal investment policies.
    Input: query — a natural-language description of the regulation, policy, or guideline.
    Returns: the most relevant document passages with their source references.
    """
    docs = retrieve(query, access_level=CALLER_ACCESS_LEVEL, top_k=3)
    if not docs:
        return "No relevant documents found in the index."
    passages = []
    for doc in docs:
        src = doc.metadata.get("source", "unknown")
        passages.append(f"[Source: {src}]\n{doc.page_content}")
    return "\n\n---\n\n".join(passages)


# ── RAG agent — combines live data tools with document retrieval ───────────────
rag_agent_tools = [get_stock_price, get_interest_rate, retrieve_policy]

rag_system_prompt = (
    "You are a senior analyst at a healthcare investment firm with access to live market data "
    "and a searchable index of regulatory, clinical, and internal policy documents. "
    "For questions about regulations, guidelines, or policy: always call retrieve_policy first. "
    "For questions about stock prices or rates: call the market data tools. "
    "Cite the source of every factual claim drawn from retrieved documents. "
    "This analysis is informational only and does not constitute investment advice."
)

rag_agent = create_react_agent(
    model=llm,
    tools=rag_agent_tools,
    prompt=rag_system_prompt,
    checkpointer=MemorySaver(),
)

print(f"RAG agent ready. Tools: {[t.name for t in rag_agent_tools]}")


# ── Helper ────────────────────────────────────────────────────────────────────
def run_rag_agent(query: str) -> str:
    tid    = str(uuid.uuid4())
    config = {"configurable": {"thread_id": tid}, "recursion_limit": 10}
    result = rag_agent.invoke({"messages": [HumanMessage(content=query)]}, config=config)
    msgs   = result["messages"]
    tools_called = [tc["name"] for m in msgs for tc in (getattr(m, "tool_calls", []) or [])]
    print(f"Tools called: {tools_called}")
    return msgs[-1].content

RAG agent ready. Tools: ['get_stock_price', 'get_interest_rate', 'retrieve_policy']


/tmp/ipykernel_7617/638677498.py:55: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  rag_agent = create_react_agent(


## Section 4 — Production Deployment: Azure AI Search

### Azure AI Search

The `retrieve_policy` tool is written against the LangChain `VectorStore` interface. Swapping FAISS for Azure AI Search requires changing only the index construction — the tool, the agent, and all downstream code remain identical.

### Azure AI Search Index Design

A production index schema includes:

| Field | Type | Purpose |
|---|---|---|
| `id` | String (key) | Unique chunk identifier |
| `content` | String (searchable) | Chunk text — used by BM25 |
| `content_vector` | Collection(Single) | Dense embedding — used by HNSW |
| `source` | String (filterable) | Document origin |
| `doc_type` | String (filterable) | regulation / clinical_guideline / internal_policy |
| `access_level` | String (filterable) | public / restricted |
| `version_date` | DateTimeOffset (filterable, sortable) | Document version for freshness filtering |

The `access_level` field is the primary security control. The OData filter `$filter=access_level eq 'public'` is applied server-side before HNSW traversal — the restricted chunks are never retrieved, never transferred, and never visible to the LLM.

### Continuous Ingestion

Documents are updated continuously in a real enterprise. The ingestion pipeline must:

1. Detect new or updated source documents (Azure Blob Storage event triggers via Event Grid).
2. Chunk and embed the updated document.
3. Delete old chunks for that document from the index (filter by `source` field, bulk delete).
4. Insert new chunks.

Step 3 before step 4 prevents stale chunks from the old version remaining in the index alongside new chunks.

In [8]:
# ── Demo 1: Regulation query — agent retrieves from index ─────────────────────
print("DEMO 1 — Regulatory query")
print("=" * 65)
r1 = run_rag_agent(
    "What is the minimum CET1 capital ratio under Basel III, "
    "and how does the current Fed Funds rate affect bank capital adequacy?"
)
print(r1)

print("\n" + "=" * 65)

# ── Demo 2: Clinical guideline + stock price — hybrid tool use ────────────────
print("\nDEMO 2 — Clinical guideline + live price")
print("=" * 65)
r2 = run_rag_agent(
    "What does the ADA 2024 guideline say about second-line therapy for type 2 diabetes, "
    "and what is the current stock price of JNJ given their diabetes portfolio?"
)
print(r2)

print("\n" + "=" * 65)

# ── Demo 3: Access control — internal policy blocked for public caller ─────────
print("\nDEMO 3 — Access control (public caller, internal policy query)")
print("=" * 65)
r3 = run_rag_agent(
    "What is the internal approval process for pharmaceutical trades exceeding $5M?"
)
print(r3)
print("\n(The internal policy document should NOT appear in the answer above.")
print(" To grant access, set CALLER_ACCESS_LEVEL = 'restricted' in Cell 8.)")

DEMO 1 — Regulatory query
Tools called: ['retrieve_policy', 'get_interest_rate']
Under Basel III, banks are required to maintain a minimum Common Equity Tier 1 (CET1) capital ratio of 4.5% of risk-weighted assets. Additionally, they must hold a capital conservation buffer of 2.5%, which effectively raises the CET1 requirement to 7.0%. Systemically important institutions may face additional surcharges of 1-3.5% depending on their classification. [Source: Basel III Capital Rules]

The current Federal Funds rate is 5.25%. A higher Fed Funds rate can impact bank capital adequacy by increasing funding costs, potentially reducing profitability and retained earnings, which are key components of CET1 capital. Additionally, higher rates may lead to changes in the valuation of assets, such as fixed-income securities, which could affect the risk-weighted asset base and, consequently, the CET1 ratio.


DEMO 2 — Clinical guideline + live price
Tools called: ['retrieve_policy', 'get_stock_price']
Ac

In [9]:
# ── Retrieval quality evaluation ──────────────────────────────────────────────
from dataclasses import dataclass

@dataclass
class RetrievalCase:
    query:           str
    expected_source: str   # the source that must appear in top-k results
    access_level:    str = "public"


def evaluate_retrieval(cases: list[RetrievalCase], top_k: int = 3) -> dict:
    """Measure hit rate and MRR over a golden retrieval dataset."""
    hits, reciprocal_ranks = 0, []

    for case in cases:
        docs    = retrieve(case.query, access_level=case.access_level, top_k=top_k)
        sources = [d.metadata["source"] for d in docs]
        hit     = case.expected_source in sources
        rank    = (sources.index(case.expected_source) + 1) if hit else None

        hits += int(hit)
        reciprocal_ranks.append(1 / rank if rank else 0)

        status = f"HIT  (rank {rank})" if hit else "MISS"
        print(f"  [{status}] {case.query[:60]:<60} → expected: {case.expected_source}")

    n       = len(cases)
    hit_rate = hits / n
    mrr      = sum(reciprocal_ranks) / n
    print(f"\nHit rate @{top_k}: {hits}/{n} = {hit_rate:.0%}")
    print(f"MRR @{top_k}:      {mrr:.3f}")
    return {"hit_rate": hit_rate, "mrr": mrr}


golden = [
    RetrievalCase("minimum CET1 capital ratio",                    "basel3_capital_rules"),
    RetrievalCase("liquidity coverage ratio stress test",           "dodd_frank_165"),
    RetrievalCase("ePHI access controls audit requirements",        "hipaa_security_rule"),
    RetrievalCase("HbA1c target type 2 diabetes first line",        "ada_2024_standards"),
    RetrievalCase("CHA2DS2-VASc anticoagulation atrial fibrillation", "esc_2020_af"),
    RetrievalCase("pharmaceutical trade approval CCO compliance",
                  "internal_policy_IB-2024-03", access_level="restricted"),
]

print("Retrieval evaluation results:")
print("-" * 65)
metrics = evaluate_retrieval(golden, top_k=3)

Retrieval evaluation results:
-----------------------------------------------------------------
  [HIT  (rank 1)] minimum CET1 capital ratio                                   → expected: basel3_capital_rules
  [HIT  (rank 1)] liquidity coverage ratio stress test                         → expected: dodd_frank_165
  [HIT  (rank 1)] ePHI access controls audit requirements                      → expected: hipaa_security_rule
  [HIT  (rank 1)] HbA1c target type 2 diabetes first line                      → expected: ada_2024_standards
  [HIT  (rank 1)] CHA2DS2-VASc anticoagulation atrial fibrillation             → expected: esc_2020_af
  [HIT  (rank 1)] pharmaceutical trade approval CCO compliance                 → expected: internal_policy_IB-2024-03

Hit rate @3: 6/6 = 100%
MRR @3:      1.000


## Summary

This lab built a complete enterprise RAG pipeline from document ingestion to a governed, evaluated production system.

**Ingestion and chunking (Cell 4)** — `RecursiveCharacterTextSplitter` at 400 characters with 10% overlap preserves semantic coherence at chunk boundaries. Each chunk carries metadata (`source`, `doc_type`, `access_level`) that drives both filtering and access control downstream. In production, replace the synthetic corpus with `PyPDFLoader` or `AzureBlobStorageContainerLoader`.

**Vector index (Cell 6)** — FAISS provides an in-process index requiring no infrastructure. The `retrieve()` function applies a metadata pre-filter before similarity search, ensuring restricted documents are never returned to unauthorised callers. A keyword-overlap re-ranker reorders candidates before returning top-k to the agent. In production, replace FAISS with `AzureSearch` — the tool interface is identical.

**Agent-driven retrieval (Cell 8)** — `retrieve_policy` is a `@tool` the agent calls on demand. This pattern lets the agent combine live market data (prices, rates) with document retrieval in a single turn, issue targeted retrieval queries for sub-questions, and skip retrieval entirely for questions answerable from tool data alone.

**Access control (Cells 6 and 10)** — `CALLER_ACCESS_LEVEL` is the single configuration point controlling document visibility. Public callers never receive restricted chunks. In production, derive this value from the authenticated user's Azure AD claims or role assignment, not from a static variable.

**Production swap (Cell 9 theory)** — Replacing FAISS with Azure AI Search requires one code change: the vector store constructor. The index schema adds `content_vector` for HNSW search, `access_level` as a filterable field for server-side access control, and `version_date` for freshness filtering. Continuous ingestion follows a delete-then-insert pattern per source document to prevent stale chunk accumulation.

**Retrieval evaluation (Cell 11)** — Hit rate @k and MRR are the primary retrieval quality metrics. Run the evaluation harness after every change to chunking strategy, embedding model, or index configuration. A regression in MRR is a signal to investigate before the change reaches production.